<a href="https://colab.research.google.com/github/Adhiaris/Grokking-Deep-Learning/blob/main/chapter_15_federated_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 15: Deep Learning on Unseen Data
### Introducing Federated Learning

Standard deep learning centralizes all training data on one server. Federated learning trains models across many devices without ever moving raw data to a central location — preserving user privacy.

## 1. The Privacy Problem

Training a model on sensitive data (emails, medical records, messages) requires sharing that data with whoever runs the training infrastructure. Federated learning separates model training from data access.

In [ ]:
approaches = {
    "Centralized Training": {
        "data location": "All data sent to one server",
        "privacy risk" : "High — server sees all raw data",
        "accuracy"     : "High — full dataset available",
    },
    "Federated Learning": {
        "data location": "Data stays on each device",
        "privacy risk" : "Low — only gradients are shared",
        "accuracy"     : "Slightly lower — no raw data sharing",
    },
}

for approach, props in approaches.items():
    print(f"\n{approach}:")
    for key, val in props.items():
        print(f"  {key:<20}: {val}")

## 2. Standard (Centralized) Training — Spam Detector

First, let's build a simple spam classifier the traditional way: all data collected centrally.

In [ ]:
import numpy as np

np.random.seed(42)

def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(o): return o * (1 - o)

data = np.array([
    [1,1,0,0,1],
    [0,1,0,0,0],
    [1,1,1,0,1],
    [0,0,0,1,0],
    [0,0,1,1,0],
    [1,0,0,0,1],
], dtype=float)

labels = np.array([[1],[1],[1],[0],[0],[0]], dtype=float)

W = 2*np.random.rand(5, 1) - 1
alpha = 0.1

print("Centralized training:")
print(f"{'Epoch':<8} {'Error':>12}")
print("-" * 22)

for epoch in range(60):
    pred  = sigmoid(data.dot(W))
    error = np.sum((pred - labels) ** 2)
    delta = (pred - labels) * sigmoid_deriv(pred)
    W    -= alpha * data.T.dot(delta)
    if epoch % 12 == 0:
        print(f"{epoch:<8} {error:>12.6f}")

## 3. Federated Learning Simulation

In federated learning each device trains locally and shares only its gradient update. The server aggregates gradients from all devices and updates the global model.

In [ ]:
import numpy as np

np.random.seed(42)

def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_deriv(o): return o * (1 - o)

device_data = [
    (np.array([[1,1,0,0,1],[0,1,0,0,0]], dtype=float), np.array([[1],[1]], dtype=float)),
    (np.array([[1,1,1,0,1],[0,0,0,1,0]], dtype=float), np.array([[1],[0]], dtype=float)),
    (np.array([[0,0,1,1,0],[1,0,0,0,1]], dtype=float), np.array([[0],[0]], dtype=float)),
]

global_W = 2*np.random.rand(5, 1) - 1
alpha    = 0.1

print(f"Number of devices: {len(device_data)}")
print(f"Each device has {device_data[0][0].shape[0]} training examples")
print()
print(f"{'Round':<8} {'Global Error':>14}")
print("-" * 25)

for round_num in range(30):
    gradients = []

    for X_local, y_local in device_data:
        local_W    = global_W.copy()
        pred       = sigmoid(X_local.dot(local_W))
        delta      = (pred - y_local) * sigmoid_deriv(pred)
        local_grad = X_local.T.dot(delta)
        gradients.append(local_grad)

    avg_gradient = np.mean(gradients, axis=0)
    global_W    -= alpha * avg_gradient

    if round_num % 6 == 0:
        all_X = np.vstack([d[0] for d in device_data])
        all_y = np.vstack([d[1] for d in device_data])
        global_error = np.sum((sigmoid(all_X.dot(global_W)) - all_y) ** 2)
        print(f"{round_num:<8} {global_error:>14.6f}")

## 4. The Gradient Leakage Attack

Sharing gradients is not perfectly private. An adversary can sometimes reconstruct training data from gradients alone. This motivates further protection.

In [ ]:
import numpy as np

print("Why shared gradients are still a privacy risk:")
print()

print("Example: If only one user has a particular rare word in their email,")
print("the gradient for that word's weight is non-zero only for that user.")
print("An attacker seeing that gradient knows that user sent a message")
print("containing that word.")
print()

secret_vocab = ["loan", "offer", "winner", "click", "free"]
alice_gradient = np.array([0.0, 0.0, 0.41, 0.0, 0.0])

inferred_words = [w for w, g in zip(secret_vocab, alice_gradient) if abs(g) > 0.01]
print(f"Alice's gradient (for weight vector): {alice_gradient}")
print(f"Inferred words in Alice's email: {inferred_words}")

## 5. Secure Aggregation with Noise (Differential Privacy)

Adding calibrated noise to gradients before sharing prevents an adversary from inferring individual data points, while still allowing the aggregate signal to be useful for training.

In [ ]:
import numpy as np

np.random.seed(0)

true_gradient  = np.array([0.5, -0.3, 0.8, 0.0, -0.1])
noise_scale    = 0.1
num_devices    = 10

noisy_gradients = []
for _ in range(num_devices):
    noise = np.random.normal(0, noise_scale, true_gradient.shape)
    noisy_gradients.append(true_gradient + noise)

aggregated = np.mean(noisy_gradients, axis=0)

print("True gradient  :", np.round(true_gradient, 4))
print("Noisy gradient (one device):", np.round(noisy_gradients[0], 4))
print(f"Aggregated mean ({num_devices} devices):", np.round(aggregated, 4))
print(f"\nMax error after aggregation: {np.max(np.abs(aggregated - true_gradient)):.4f}")
print("Averaging cancels noise while preserving the true gradient signal.")